# 12 — UC governance primitives for PawShield

This notebook lands PawShield's Unity Catalog governance artefacts:
the SQL for a column mask on `customers.email` and the
inference-table PII-block audit queries.

**What this notebook creates:**

* §1 — `${catalog}.pawshield.mask_email`: a SQL UDF that conditionally
  masks email unless the caller is in the `compliance` account group,
  attached to `customers.email` as a column mask.
* §2 — `${catalog}.pawshield.user_state_map` (lookup table) +
  `${catalog}.pawshield.current_user_state()` (UDF) +
  `${catalog}.pawshield.filter_by_state` (row filter): scope
  `customers` reads so admins see all rows and everyone else sees
  only rows for their assigned state. Illustrative — replace the
  lookup table with whatever per-user authorisation table your
  workspace already maintains.
* §3 — Two audit queries: one aggregate against
  `system.ai_gateway.usage`, one per-row against the PolicyPal endpoint's
  inference table `${catalog}.monitoring.policypal_payload`.

## How to run it

1. Set the `catalog` widget to the Unity Catalog you used for PawShield
   (the book uses `genaicert`).
2. Run all cells. The notebook runs on Databricks default serverless
   compute; no cluster creation needed.
3. §3 returns rows only after the PolicyPal endpoint has served
   requests (see `c1001-deploy-policypal`). On a fresh
   workspace both queries return zero rows — expected.

## What this notebook does NOT do

* No destructive operations on `customers`. The GDPR right-to-be-forgotten
  flow (`DELETE` → `REORG TABLE … APPLY (PURGE)` →
  `VACUUM`) is shown in the chapter prose only — running it here would
  delete rows the rest of the book depends on.
* No `compliance` or `state_admins` account groups are created. The
  mask and row filter attach cleanly regardless; with empty groups every
  caller sees the restricted form, which is the right default for a
  teaching workspace.

In [0]:
dbutils.widgets.text("catalog", "genaicert")
catalog = dbutils.widgets.get("catalog")
print(f"Catalog: {catalog}")
spark.sql(f"USE CATALOG {catalog}")
spark.sql("USE SCHEMA pawshield")

Catalog: genaicert


DataFrame[]

## 1. Column mask on `customers.email`

The mask routes every read of `customers.email` through `mask_email`.
Callers in the `compliance` account group see the raw value; everyone
else sees first-letter + `***@` + domain (e.g.,
`sarah.chen@example.com` → `s***@example.com`). No application-layer
code changes — the rewrite happens inside UC.

In [0]:
%sql
CREATE OR REPLACE FUNCTION pawshield.mask_email(email STRING)
RETURNS STRING
RETURN CASE
    WHEN is_account_group_member('compliance') THEN email
    ELSE concat(left(email, 1), '***@', split_part(email, '@', 2))
END;

In [0]:
%sql
ALTER TABLE pawshield.customers
ALTER COLUMN email SET MASK pawshield.mask_email;

Verify: query `customers.email` and confirm the masked form returns.
Unless the running identity is in the `compliance` group, values look
like `s***@example.com`.

In [0]:
%sql
SELECT customer_id, first_name, last_name, email
FROM pawshield.customers
LIMIT 5;

customer_id,first_name,last_name,email
CUST-CHEN-001,Sarah,Chen,s***@gmail.com
CUST-RODRIGUEZ-002,Carmen,Rodríguez,c***@example.com
CUST-WHITFIELD-003,Eleanor,Whitfield,e***@nyu.edu
CUST-PARK-004,Min-jun,Park,m***@gmail.com
CUST-JOHNSON-005,Tasha,Johnson,t***@yahoo.com


## 2. Row filter scoping `customers` reads to a state slice

The row filter restricts which *rows* the caller can see (column mask
restricts which *values*; the two primitives compose). Admins in the
`state_admins` account group see every row; everyone else sees only
the rows whose `address_state` matches their assigned state, looked
up via a small `user_state_map` table + a `current_user_state()` UDF.

**Note on column naming.** The column listed in `SET ROW FILTER … ON (…)`
is passed *positionally* to the row-filter function's parameter — only the
*types* must match, not the names. PawShield's column is `address_state`
(not bare `state`); the UDF signature reuses that same name purely for
readability, and `ON (address_state)` binds it positionally.

**Production shape.** `user_state_map` is illustrative — a lookup
table mapping `current_user()` to the caller's assigned state.
Replace with whatever per-user authorisation table your workspace
already maintains. On a fresh workspace the table is empty, so
non-admin callers see zero rows until a row for their identity is
inserted — the safe default for a teaching demo.

In [0]:
%sql
-- Lookup table: each user maps to one assigned state. Empty by
-- default; non-admin callers see no customer rows until a row is
-- inserted for their `current_user()` identity.
CREATE TABLE IF NOT EXISTS pawshield.user_state_map (
    user_email STRING NOT NULL,
    address_state STRING NOT NULL
);

In [0]:
%sql
-- Helper UDF resolving the caller's assigned state from user_state_map.
-- Returns NULL when no row matches; the row filter treats NULL as
-- "no state assigned" (no rows visible) for non-admin callers.
CREATE OR REPLACE FUNCTION pawshield.current_user_state()
RETURNS STRING
RETURN (
    SELECT address_state
    FROM pawshield.user_state_map
    WHERE user_email = current_user()
    LIMIT 1
);

In [0]:
%sql
CREATE OR REPLACE FUNCTION pawshield.filter_by_state(address_state STRING)
RETURNS BOOLEAN
RETURN is_account_group_member('state_admins')
    OR address_state = pawshield.current_user_state();

In [0]:
%sql
ALTER TABLE pawshield.customers
SET ROW FILTER pawshield.filter_by_state ON (address_state);

Verify: the row count drops to the TX subset unless you're in
`state_admins`.

In [0]:
%sql
SELECT COUNT(*) AS visible_rows FROM pawshield.customers;

visible_rows
0


## 3. Inference-table queries — assurance the PII filter is firing

Two SQL surfaces feed the PII-block audit dashboard:

* `system.ai_gateway.usage` — one row per AI Gateway request with
  `endpoint_name`, `event_time`, `status_code`, per-request token
  counts. The doc-grounded aggregate surface for cross-endpoint
  monitoring; `status_code = 400` is the documented HTTP signature of
  an enforce-mode guardrail block.
* `${catalog}.monitoring.policypal_payload` — the PolicyPal endpoint's
  inference table; per-row status + full request/response JSON for
  forensic replay.

**PyFunc constraint to remember.** 
PolicyPal is a Custom Model Serving
(PyFunc) endpoint, and AI Gateway guardrails are unsupported on PyFunc
per the Model Serving endpoint-type support matrix. `status_code = 400` rows
on `policypal-chain` are genuine client errors (malformed requests),
*not* guardrail blocks; rate-limit throttling surfaces as `429`, not
 `400`. To see real guardrail-block rows, point the same query shape at
an FM-API pay-per-token or External-Models endpoint where guardrails
fire.

In [0]:
%sql
-- §3.1 — Per-day block counts at the AI Gateway layer (aggregate view)
SELECT
  endpoint_name,
  DATE(event_time)                                   AS day,
  COUNT(*)                                           AS requests,
  SUM(CASE WHEN status_code = 400 THEN 1 ELSE 0 END) AS blocked
FROM system.ai_gateway.usage
WHERE endpoint_name = 'policypal-chain'
GROUP BY 1, 2
ORDER BY 2 DESC;

endpoint_name,day,requests,blocked


In [0]:
%sql
-- §3.2 — Per-row detail for forensic replay (the PolicyPal inference table)
SELECT
  request_date,
  request_time,
  status_code,
  execution_duration_ms,
  SUBSTRING(CAST(request  AS STRING), 1, 200) AS request_preview,
  SUBSTRING(CAST(response AS STRING), 1, 200) AS response_preview
FROM monitoring.policypal_payload
WHERE status_code = 400
ORDER BY request_time DESC
LIMIT 20;

request_date,request_time,status_code,execution_duration_ms,request_preview,response_preview


## 4. Teardown (Important)

Drop the mask + row filter to restore `customers` to its unrestricted
shape. Otherwise you won't be able to select any data from customers table unless you set up user groups or mapping tables properly as described.

```sql
ALTER TABLE ${catalog}.pawshield.customers ALTER COLUMN email DROP MASK;
DROP FUNCTION ${catalog}.pawshield.mask_email;

ALTER TABLE ${catalog}.pawshield.customers DROP ROW FILTER;
DROP FUNCTION ${catalog}.pawshield.filter_by_state;
DROP FUNCTION ${catalog}.pawshield.current_user_state;
DROP TABLE  ${catalog}.pawshield.user_state_map;
```

In [0]:
%sql
ALTER TABLE pawshield.customers ALTER COLUMN email DROP MASK;
DROP FUNCTION if exists pawshield.mask_email;

ALTER TABLE pawshield.customers DROP ROW FILTER;
DROP FUNCTION if exists pawshield.filter_by_state;
DROP FUNCTION if exists pawshield.current_user_state;
DROP TABLE if exists pawshield.user_state_map;